<a href="https://colab.research.google.com/github/m-edal/Earth-Env-DS-MSc-Course/blob/main/labs/W13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bias Correction for MUSICA Simulation with AutoML

**Lab Duration:** ~1-2 hours

## Overview

In this lab, you will learn how to apply **AutoML** techniques to perform bias correction of **PM2.5** concentrations — improving the accuracy of model simulations by learning systematic discrepancies between model outputs and observations.

We will work with **PM2.5** data derived from the Multi-Scale Infrastructure for Chemistry and Aerosol (MUSICA) simulations. This dataset includes hourly MUSCIAv0 simulations from 20181120-20181130. By comparing modelled PM2.5 with observational data. AutoML frameworks can automatically select and optimize machine learning models to correct biases across different regions and time periods.

## Objective

The objective of this lab is to develop an **AutoML-based bias correction model** for PM2.5 that:
- Learns the relationship between model outputs and observations
- Reduces systematic biases in simulated PM2.5 concentrations
- Improves the representation of temporal variability and extreme pollution events
- Provides a scalable and automated framework for air quality model post-processing

---

### Install AutoML Library (FLAML)

In [ ]:
!pip install flaml

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from flaml import AutoML

### Data location
- MUSICAv0 simulation: './data/MUSICA_bias_correction_20181120_20181130.csv': https://github.com/m-edal/Earth-Env-DS-MSc-Course/blob/main/labs/data/MUSICA_bias_correction_20181120_20181130.csv

In [ ]:
# ==============================
# 1. Read the data
# ==============================

df = pd.read_csv(
    "~/PATH/TO/MUSICA_bias_correction_20181120_20181130.csv"
)

print("Number of rows:", len(df))
print("Available columns:")
print(df.columns.to_list())



In [ ]:
# ==============================
# 2. Define target and baseline model output
# ==============================

obs_col = "PM2.5 (Hourly measured)"   # Observed PM2.5
raw_col = "PM25"                     # Raw MUSICA PM2.5 prediction
time_col = "datetime"


# ==============================
# 3. Select input features
# ==============================

# Basic meteorological variables
meteo_cols = [
    "CLOUD", "FSDS", "T", "U", "V",
    "OMEGA500", "PBLH", "RELHUM", "PRECT"
]

# Emission / chemistry-related variables
cams_cols = [c for c in df.columns if c.startswith("CAMSv51_")]
qfed_cols = [c for c in df.columns if c.startswith("QFED_")]
vsl_cols  = [c for c in df.columns if c.startswith("vsl_")]

# Combine all feature columns
feature_cols = meteo_cols + cams_cols + qfed_cols + vsl_cols

print("Number of features:", len(feature_cols))
print("Feature columns:")
print(feature_cols)



In [ ]:
# ==============================
# 4. Basic data cleaning
# ==============================

df = df.copy()

# Convert datetime column
df[time_col] = pd.to_datetime(df[time_col])

# Drop rows with missing values in target, raw model output, or features
df = df.dropna(subset=[obs_col, raw_col] + feature_cols)

# Define the residual target
# residual = observation - raw model prediction
df["residual"] = df[obs_col] - df[raw_col]

print("Rows after cleaning:", len(df))

In [ ]:
# ==============================
# 5. Prepare X and y
# ==============================

X = df[feature_cols]
y = df["residual"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X,
    y,
    df,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


# ==============================
# 6. Train the AutoML model
# ==============================

automl = AutoML()

automl_settings = {
    "time_budget": 60,          # Maximum training time in seconds
    "metric": "rmse",           # Optimisation metric
    "task": "regression",       # Regression task
    "estimator_list": ["lgbm"],  # Use LightGBM only for simplicity
    "seed": 42,
    "verbose": 1,
}

automl.fit(
    X_train=X_train,
    y_train=y_train,
    **automl_settings
)

print("Best model:", automl.best_estimator)
print("Best config:", automl.best_config)

# ==============================
# 7. Predict residuals on the test set
# ==============================

residual_pred = automl.predict(X_test)

# Corrected PM2.5 = raw MUSICA PM2.5 + predicted residual
pm25_raw = df_test[raw_col].values
pm25_obs = df_test[obs_col].values
pm25_corr = pm25_raw + residual_pred

In [ ]:
# ==============================
# 8. Define evaluation functions
# ==============================

def rmse(y_true, y_pred):
    """Calculate root mean squared error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def nmb(y_true, y_pred):
    """Calculate normalized mean bias in percentage."""
    return np.sum(y_pred - y_true) / np.sum(y_true) * 100


# ==============================
# 9. Evaluate raw and corrected PM2.5
# ==============================

results = {
    "RMSE_raw": rmse(pm25_obs, pm25_raw),
    "RMSE_corrected": rmse(pm25_obs, pm25_corr),

    "MAE_raw": mean_absolute_error(pm25_obs, pm25_raw),
    "MAE_corrected": mean_absolute_error(pm25_obs, pm25_corr),

    "R2_raw": r2_score(pm25_obs, pm25_raw),
    "R2_corrected": r2_score(pm25_obs, pm25_corr),

    "NMB_raw_%": nmb(pm25_obs, pm25_raw),
    "NMB_corrected_%": nmb(pm25_obs, pm25_corr),
}

results_df = pd.DataFrame([results])

print("Evaluation results:")
print(results_df.to_string(index=False))



In [ ]:
# ==============================
# 10. Save corrected results
# ==============================

df_test_out = df_test.copy()
df_test_out["residual_pred"] = residual_pred
df_test_out["PM25_corrected"] = pm25_corr

df_test_out.to_csv(
    "corrected_test.csv",
    index=False
)

print("Saved corrected test results.")